# Tutorial 14: Reinforcement Learning (Part I)

In this tutorial, we will use two separate Reinforcement Learning approaches to perform a model-free pendulum swing-up in real time.

**Pre-requisites**

Knowledge of the fundamentals of RL, as well as Dynamic Programming.

**Goals**

Demonstrating the properties of DDPG and SAC. Showcasing the capabilities of CloudPendulum Gym.

This notebook is organized as follows:

    1. Recap: Reinforcement Learning methods and their classification
    2. Deep Deterministic Policy Gradient
    3. Soft Actor-Critic



# 1. Reinforcement Learning recap

Fundamentally, Reinforcement Learning consists in training agents using trial and error.
The **agent** interacts with an **environment** by performing **actions**, which modify the **state** of the environment. As the agent interacts with the environment, it perceives **observations** and is given **rewards**, which guide its decisions. This happens because the goal of the agent is to maximize the rewards it accumulates. The cummulative reward receives the name **return**.

As the agent learns, its **policy** will take shape. The role of the policy is to determine the actions of the agent depending on its observations. The central problem in Reinforcement Learning can be written as:

$$\pi^* = \text{argmax}_\pi E \left[ \sum^T_{t=0}r_t(s_t,a_t) \right]$$

where $\pi$ is the policy and $\pi^*$ denotes the optimal policy. $E \left[ \sum^T_{t=0}r_t(s_t,a_t) \right]$ is the expected cummulative reward during a trajectory, with $r_t$ being the reward at time $t$ given state $s_t$ and action $a_t$. This may remind you of Bellman's equation:

$$V^*(s_t) = \max_a E \left[r(s_t, a_t) + \gamma V^*(s_{t+1}) \right]$$

In this version, the optimal cost-to-go (in this case, the reward) at one state, $V^*(s_t)$ is the maximum return you can expect if you find yourself at $s_t$ during a trajectory. Using Bellman's optimality principle we also reach the Q-function, which defines the optimal state-action pair as the one that provides the highest expected return out of all the feasible actions at a given state.

$$Q^*(s_t,a_t) = E \left[r(s_t, a_t) + \gamma \max_{a_{t+1}}Q^*(s_{t+1},a_{t+1}) \right] $$

### Types of RL algorithms

A very common way to classify RL algorithms is by putting them in the model-based or model-free categories. In this tutorial, we will cover two model-free approaches, Deep Deterministic Policy Gradient (DDPG) and Soft Actor-Critic (SAC)

<div style="display: flex; justify-content: space-around;">
    <div><img src="media/rl_algorithms_9_15.svg" width="800"></div>
</div>
Source: https://spinningup.openai.com

Another way in which we differentiate between algorithms is by dividing them between on- and off-policy. On-policy algorithms refine the policy by testing it and learning from the data that the policy generates, whereas off-policy algorithms use pre-existing data. One of the advantages of off-policy algorithms is that they can re-use training data.

# 2. Deep Deterministic Policy Gradient

The first method we will look at today is DDPG. The fundamental idea behind it is to have our algorithm learn the Q-function, as well as the optimal policy at the same time. This method is designed to deal exclusively with continuous action spaces. It deals with those by assuming that $Q^*$ is differentiable with respect to $a$. This is a Deep Learning method because it uses neural networks to approximate the Q-function and policy.

These two networks are called actor for the network that learns the policy; and critic for the network that learns the Q-function. Additionally, we keep a weighted average of both networks around as time goes on, which we call target actor and target critic networks, respectively. The goal of the critic network is to minimize $L$, the expected difference mean-squared Bellman error.

  $$ L(\phi, \mathcal{D}) = {E}_{(s,a,r,s') \sim \mathcal{D}}\left[(Q_\phi(s,a) - y)^2\right] $$

  where $y = r+ \gamma(1-d) \max_{a_{t+1}} Q_\phi(s_{t+1}, a_{t+1})$

Before starting our DDPG loop, we need to define a few variables:

- The reward function $r(s,a)$, which we want our policy to maximize.

- A set of transitions $\mathcal{D}$. $\mathcal{D}$ stores the states, actions, rewards, next states, and $d$, which indicates whether the final state is reached.

- Some tuneable parameters like the learning rate $\eta$ or the target network update rate $\rho$.

The typical DDPG training loop works as follows:

- The agent observes the state, computes the action, and executes it.

  $$a = \mu_\theta(s) + \mathcal{N}$$

  where $\mu_\theta$ is the actor network with parameters $\mu$ and $\mathcal{N}$ is an added random noise. Since the policy is deterministic, this allows us to explore a wider range of actions instead of always ascending the gradient in the exact same way.

- Observe the reward, next state, and whether the $s_{t+1}$ is terminal. Store this data in $\mathcal{D}$. If the next state is terminal, reset the state.

- Sample a batch of transitions, $B$ from $\mathcal{D}$ and compute the critic. When we update the critic, we use the target networks for stability.

  $$ y(r,s_{t+1},d) = r+ \gamma(1-d) \max_{a_{t+1}} Q_{\phi_{targ}}(s_{t+1}, \mu_{\theta_{targ}}(s_{t+1}))$$

- Update the critic network.

  $$ \phi = \phi -\eta \nabla_\phi \frac{1}{\left| B \right|} \sum_{(s,a,r,s_{t+1},d)\in B} (Q_\phi(s,a)-y(r,s_{t+1},d))^2$$

- Update the actor network.

  $$ \theta = \theta +\eta \nabla_\theta \frac{1}{\left| B \right|} \sum_{s\in B} Q_\phi(s,\mu_\theta(s)) $$

- Update the target networks.

  $$ \phi_{targ} = \rho \phi_{targ} + (1-\rho)\phi$$

  $$ \theta_{targ} = \rho \theta_{targ} + (1-\rho)\theta$$

Then, we run this loop successively until convergence.

## 2.2. DDPG training in simulation.

To start with our implementation of DDPG, we will import some necessary modules

In [1]:
USER_TOKEN = "" # Write your token here
username = "" # Write your username here

import sys
sys.path.insert(0, '/home/jupyter-'+username+'/.local/lib/python3.12/site-packages')

import google.protobuf

import os
import numpy as np
import matplotlib.pyplot as plt

from ddpg.ddpg_hw import ddpg_trainer
from IPython.display import HTML, clear_output, display

I0000 00:00:1778431049.092276  192297 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778431049.279132  192297 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778431054.414992  192297 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/mi

First, we will train our DDPG implementation using a simulated version of our pendulums. For that, we will input its properties in the cell below.

In [2]:
mass = 0.06
length = 0.1
damping = 0.0004
torque_limit = 0.02
coulomb_fric = 0.0
inertia = mass*length**2
gravity = 9.81

We will also need to define the parameters and reward function of our algorithm.

In [3]:
# environment parameters
dt = 0.05 # changed from 0.01
integrator = "runge_kutta"
max_steps = 200
reward_type = "combined_reward"
target = [np.pi, 0]
target_epsilon = [0.05, 0.05] # Changed from 0.05
random_init = "everywhere"

# training parameters
n_episodes = 100
batch_size = 64*4*3
validate_every = 10 # changed from 20
validation_reps = 10
validation_limit = 0 # changed from -1800
train_every_steps = 1 # changed from 1
state_representation = 3
replay_buffer_size = 1000 # changed from 50000
actor = None  # use default agent
critic = None  # use default critic
discount = 0.99
actor_lr = 0.0005 # changed from 0.0005 
critic_lr = 0.001 # Changed from 0.001
tau = 0.005 # changed from 0.005

def swingup_reward(self, observation, action):
    """
    Calculate the reward for the pendulum for swinging up to the instable
    fixpoint. The reward function is selected based on the reward type
    defined during the object inizialization.

    Parameters
    ----------
    state : array-like
        the observation that has been received from the environment

    Returns
    -------
    reward : float
        the reward for swinging up

    Raises
    ------
    NotImplementedError
        when the requested reward_type is not implemented

    """
    reward = None
    pos = observation[0] % (2*np.pi)
    pos_diff = self.target[0] - pos
    pos_diff = np.abs((pos_diff + np.pi) % (np.pi * 2) - np.pi)
    vel = np.clip(observation[1], self.low[-1], self.high[-1])
    vel_diff = np.abs(self.target[1] - vel)
    
    reward = (-(pos_diff)**2.0 -
             0.01*(vel_diff)**2.0 - # change from 0.1
              0.01*action**2.0) + 5*np.exp(-pos_diff**2 / (2 * 0.25**2)) # change from 0.01
    if pos_diff < self.state_target_epsilon[0] and abs(vel_diff)< self.state_target_epsilon[1]:
        reward += 50
    if pos_diff> 2*np.pi/3:
        reward = -10
    return reward

In [4]:
save_dir = "log_data/ddpg_training"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)



trainer = ddpg_trainer(batch_size=batch_size,
                       validate_every=validate_every,
                       validation_reps=validation_reps,
                       train_every_steps=train_every_steps)

trainer.init_pendulum(mass=mass,
                      length=length,
                      inertia=inertia,
                      damping=damping,
                      coulomb_friction=coulomb_fric,
                      gravity=gravity,
                      torque_limit=torque_limit)

trainer.init_environment(dt=dt,
                         integrator=integrator,
                         max_steps=max_steps,
                         reward_type=reward_type,
                         state_representation=state_representation,
                         validation_limit=validation_limit,
                         target=target,
                         state_target_epsilon=target_epsilon,
                         random_init=random_init)

trainer.init_agent(replay_buffer_size=replay_buffer_size,
                   actor=actor,
                   critic=critic,
                   discount=discount,
                   actor_lr=actor_lr,
                   critic_lr=critic_lr,
                   tau=tau)

rewards, actor_losses, critic_losses = trainer.train(n_episodes=n_episodes,
                                                     verbose=True)

trainer.save(save_dir)

/opt/jupyterhub/venv/lib/python3.12/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
E0000 00:00:1778431061.486337  192297 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778431061.486900  192441 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1778431061.578508  192297 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platf

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ multiply (Multiply)             │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,073 (262.00 KB)

 Trainable params: 67,073 (262.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 16)        │         64 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │        544 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │         64 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dense_7[0][0],    │
│ (Concatenate)       │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 256)       │     16,640 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 256)       │     65,792 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 1)         │        257 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 83,361 (325.63 KB)

 Trainable params: 83,361 (325.63 KB)

 Non-trainable params: 0 (0.00 B)

None
Episode: 0, Steps: 199, Reward: -2000, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [ 0.06685586 -0.67559576], Success: False
Episode: 1, Steps: 199, Reward: -1975.77, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [0.0275497  0.09665839], Success: False
Episode: 2, Steps: 199, Reward: -1891.85, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [ 0.10508482 -0.1044274 ], Success: False
Episode: 3, Steps: 199, Reward: -2000, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [-0.04892807  0.43260938], Success: False
Episode: 4, Steps: 199, Reward: -1884.18, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [ 0.05875632 -0.37871718], Success: False
Episode: 5, Steps: 199, Reward: -2000, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [-0.01517045  0.0015683 ], Success: False
Episode: 6, Steps: 199, Reward: -2000, Actor Loss: 0.0, Critic Loss: 0.0, Final State: [ 0.13386302 -0.05635547], Success: False
Episode: 7, Steps: 199, Reward: -1963.93, Actor Loss: 0.0, Critic Loss: 0.0, Final St

When our training is done, we can see the result on a simulated pendulum.

# 3. Soft Actor-Critic

DDPG, as its name indicates, produces a deterministic policy. This means that a certain observation will always result in the same action. To mitigate this, we added some noise, which made the algorithm explore more solutions. However, this noise is hand-tuned and susceptible to performance issues depending on the environment. One of the ways to mitigate this is by using a stochastic policy instead. Deterministic policies will always provide the same action in response to a state. Stochastic policies put out a probability distribution as a function of the observed state.

When it comes to SAC, in addition to maximizing the return, we also seek to maximize entropy, that is, the randomness of the policy. Entropy is a measure of the randomness of a probability distribution. The more uncertain the action given a state, the higher the entropy. In mathematical terms, this updates our RL problem to:

$$\pi^* = \text{argmax}_\pi E \left[ \sum^\infty_{t=0} \gamma^t \left( R(s_t,a_t,s_{t+1}) + \alpha H(\pi(\cdot \mid s_t)) \right) \right]$$

where $R$ is the reward received for taking action $a_t$ at state $s_t$ and it resulting in a transition to state $s_{t+1}$. $H(\pi(\cdot \mid s_t)$ is the entropy associated to the policy at state $s_t$, which is calculated as $H(\pi(\cdot \mid s_t) = E \left[ -\log \pi(a \mid s_t) \right]$. $\alpha$ is a temperature parameter controlling the trade-off between maximizing the reward or the entropy. 

The Q-function and cost-to-go functions are also updated to include an entropy-maximizing term.

$$ Q^\pi(s,a) = E \left[ \sum^\infty_{t=0} \gamma^t R(s_t,a_t,s_{t+1} + \alpha \sum^\infty_{t=0} \gamma^t H(\pi(\cdot \mid s_t))\right]$$

$$ V^\pi (s_t) = E \left[ Q^(s_t,a_t)\right] + \alpha H(\pi(\cdot \mid s_t))$$

The resulting Bellman equation is then:

$$Q^\pi(s_t, a_t) = E\left[R(s_t, a_t, s_{t+1}) + \gamma\left(Q^\pi(s_{t+1}, a_{t+1}) + \alpha H (\pi(\cdot \mid s_{t+1}))\right)\right]$$

The typical SAC training loop is similar to DDPG, with three key differences: the policy is stochastic, two critic networks are used instead of one, and an entropy term is added to the objective. The loop works as follows:

- The agent observes the state and samples an action directly from the policy, without added noise:
  $$\tilde{a} \sim \pi_\theta(\cdot \mid s)$$
  Exploration is no longer a concern here, as the stochastic policy naturally explores by sampling different actions in the same state.
  
- Observe the reward, next state, and whether $s_{t+1}$ is terminal. Store $(s, a, r, s_{t+1}, d)$ in $\mathcal{D}$. If the next state is terminal, reset the state.


- Sample a batch of transitions $B$ from $\mathcal{D}$ and compute the target for both critics, using the minimum of the two target critics to reduce overestimation bias:
$$y(r, s_{t+1}, d) = r + \gamma(1-d)\left(\min_{i=1,2} Q_{\phi_{\text{targ},i}}(s_{t+1}, \tilde{a}_{t+1}) - \alpha \log \pi_\theta(\tilde{a}_{t+1} \mid s_{t+1})\right), \quad \tilde{a}_{t+1} \sim \pi_\theta(\cdot \mid s_{t+1})$$

- Update both critic networks independently against the shared target $y$.
$$\phi_i = \phi_i - \eta \nabla_{\phi_i} \frac{1}{|B|} \sum_{(s,a,r,s_{t+1},d) \in B} (Q_{\phi_i}(s,a) - y(r,s_{t+1},d))^2 \quad \text{for } i=1,2$$
- Update the actor network.
 $$\theta = \theta + \eta \nabla_\theta \frac{1}{|B|} \sum_{s \in B} \left(\min_{i=1,2} Q_{\phi_i}(s, \tilde{a}_\theta(s)) - \alpha \log \pi_\theta(\tilde{a}_\theta(s) \mid s)\right)$$

- Update the target networks, as in DDPG:
 $$\phi_{\text{targ},i} = \rho \phi_{\text{targ},i} + (1-\rho)\phi_i \quad \text{for } i=1,2$$
 Note that SAC has no target actor — the current actor $\pi_\theta$ is used directly when computing targets.

Then, we run this loop successively until convergence.